###### 方案：Transformer 排序 + LightGBM Ranker 堆叠
- 第一阶段：按时间顺序做 3 折 OOF，用每折的 Transformer 给验证折打分，得到无偏的 `Transformer_Score`。
- 第二阶段：在全部训练集上训练同一个 Transformer，按验证集 `final_score` 早停并保存最佳权重。
- 第三阶段：把 OOF 阶段的截面特征与 `Transformer_Score` 拼接，训练 `LGBMRanker`（lambdarank）做二次排序；高维特征用小学习率、浅树、行列采样与早停控制过拟合。
- 训练结束后输出特征重要性，用于检查时序分数是否提供了增量信息。

## 01基础配置
#### 超参 + 路径
- 定义 sequence_length=60 ：每只股票用过去 60 个交易日的特征序列作为输入。
- feature_num='158+39' ：决定后续特征工程用哪套特征列。
- config 里是 Transformer 结构参数（ d_model/nhead/layers/... ）、训练参数（ batch_size/num_epochs/lr ）、梯度裁剪、early stopping（ patience ）、top-k 相关权重（ top5_weight/base_weight/pairwise_weight ）以及输出目录、数据目录。


In [2]:
# 配置参数
sequence_length = 60
feature_num = '158+39'
config = {
    'sequence_length': sequence_length,   # 使用过去60个交易日的数据（排序任务可以用稍短的序列）
    'd_model': 256,          # Transformer输入维度
    'nhead': 4,             # 注意力头数量
    'num_layers': 2,        # Transformer层数
    'dim_feedforward': 512, # 前馈网络维度
    'batch_size': 4,        # 排序任务batch_size可以小一些，因为每个batch包含更多股票
    'num_epochs': 40,       # 排序任务可能需要更多epochs
    'learning_rate': 3e-5,  # 稍微降低学习率
    'dropout': 0.15,
    'feature_num': feature_num,
    'max_grad_norm': 3.0,

    'pairwise_weight': 0.0, # 配对损失权重
    'base_weight': 1.0, # 非top-k样本权重
    'top5_weight': 3.0, # top-5样本权重（应大于base_weight）

    'patience':10,

    'output_dir': f'./model/{sequence_length}_{feature_num}',
    'data_path': './data',
}

## 02模型加载
#### 定义排序用 Transformer
- PositionalEncoding ：标准正弦位置编码，加到序列特征上。
- FeatureAttention ：对时间维（seq_len）做注意力加权汇聚，把 [batch*num_stocks, seq_len, d_model] 压成 [batch*num_stocks, d_model] 。
- CrossStockAttention ：在“同一天的一组股票”之间做多头注意力交互，输入形状 [batch, num_stocks, d_model] ，并支持 key_padding_mask 来忽略 padding 的股票。
- StockTransformer.forward(src, masks=None) ：
- 先把 [batch, num_stocks, seq_len, feature_dim] reshape 成 [batch*num_stocks, seq_len, feature_dim] 做输入投影 + 时序 Transformer Encoder；
- 再用 FeatureAttention 把每只股票的序列聚合成向量；
- reshape 回股票维度后做 CrossStockAttention （股票间交互）；
- 最后 ranking_layers + score_head 输出每只股票一个分数，形状 [batch, num_stocks] 。
   这个分数用于“在同一天内把股票排序”。

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

# 位置编码模块
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)
        
class CrossStockAttention(nn.Module):
    """股票间交互注意力模块"""
    def __init__(self, d_model, nhead, dropout=0.1):
        super(CrossStockAttention, self).__init__()
        self.cross_attention = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, stock_features, key_padding_mask=None):
        """
        stock_features: [batch, num_stocks, d_model]
        key_padding_mask: [batch, num_stocks]
            True 表示该位置是 padding，需要被 attention 忽略
            False 表示真实股票
        """
        attended, _ = self.cross_attention(
            stock_features,
            stock_features,
            stock_features,
            key_padding_mask=key_padding_mask
        )
        output = self.norm(stock_features + self.dropout(attended))
        return output

class FeatureAttention(nn.Module):
    """特征注意力模块"""
    def __init__(self, d_model, dropout=0.1):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
            nn.Softmax(dim=1)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x: [batch*num_stocks, seq_len, d_model]
        attention_weights = self.attention(x)  # [batch*num_stocks, seq_len, 1]
        attended = torch.sum(x * attention_weights, dim=1)  # [batch*num_stocks, d_model]
        return self.dropout(attended)

class StockTransformer(nn.Module):
    def __init__(self, input_dim, config, num_stocks, emb_dim=16):
        super(StockTransformer, self).__init__()
        self.model_type = 'RankingTransformer'
        self.config = config
        self.num_stocks = num_stocks
        
        # 输入投影层
        self.input_proj = nn.Linear(input_dim, config['d_model'])
        self.pos_encoder = PositionalEncoding(config['d_model'], config['dropout'], config['sequence_length'])
        
        # 时序特征提取
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config['d_model'],
            nhead=config['nhead'],
            dim_feedforward=config['dim_feedforward'],
            dropout=config['dropout'],
            batch_first=True
        )
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config['num_layers'])
        
        # 特征注意力
        self.feature_attention = FeatureAttention(config['d_model'], config['dropout'])
        
        # 股票间交互注意力
        self.cross_stock_attention = CrossStockAttention(config['d_model'], config['nhead'], config['dropout'])
        
        # 排序特异性层
        self.ranking_layers = nn.Sequential(
            nn.Linear(config['d_model'], config['d_model']),
            nn.LayerNorm(config['d_model']),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(config['d_model'], config['d_model'] // 2),
            nn.LayerNorm(config['d_model'] // 2),
            nn.ReLU(),
            nn.Dropout(config['dropout'])
        )
        
        # 最终排序分数输出
        self.score_head = nn.Sequential(
            nn.Linear(config['d_model'] // 2, config['d_model'] // 4),
            nn.ReLU(),
            nn.Dropout(config['dropout'] * 0.5),
            nn.Linear(config['d_model'] // 4, 1)
        )
        
        # 初始化权重
        self._init_weights()
        
    def _init_weights(self):
        """初始化模型权重"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    

    def forward(self, src, masks=None):
        """
        src: [batch, num_stocks, seq_len, feature_dim]
        masks: [batch, num_stocks]
            1 表示真实股票
            0 表示 padding 股票
        """
        batch_size, num_stocks, seq_len, feature_dim = src.size()
        
        # [batch*num_stocks, seq_len, feature_dim]
        src_reshaped = src.view(batch_size * num_stocks, seq_len, feature_dim)
        
        # 输入投影和位置编码
        src_proj = self.input_proj(src_reshaped)
        src_proj = self.pos_encoder(src_proj)
        
        # 时序特征提取
        temporal_features = self.temporal_encoder(src_proj)
        
        # 特征注意力聚合
        aggregated_features = self.feature_attention(temporal_features)
        
        # [batch, num_stocks, d_model]
        stock_features = aggregated_features.view(batch_size, num_stocks, -1)
        
        # MultiheadAttention 的 key_padding_mask:
        # True 表示要屏蔽，False 表示保留
        key_padding_mask = None
        if masks is not None:
            key_padding_mask = masks == 0

        # 股票间交互注意力
        interactive_features = self.cross_stock_attention(
            stock_features,
            key_padding_mask=key_padding_mask
        )
        
        # [batch*num_stocks, d_model]
        interactive_features = interactive_features.view(batch_size * num_stocks, -1)
        
        # 排序层
        ranking_features = self.ranking_layers(interactive_features)
        
        # 输出分数
        scores = self.score_head(ranking_features)
        
        # [batch, num_stocks]
        output = scores.view(batch_size, num_stocks)
        
        return output


## 03特征工程
#### 158+39 特征 + 构造按日排序样本
- engineer_features_158plus39(df) ：先算 158 个 Alpha 类特征（ engineer_features ），再算 39 个技术指标（ engineer_features_39 ），合并后去重并 inf/NaN -> 0 。
- engineer_features_39(df) ：用 TA-Lib 算 SMA/EMA/MACD/RSI/KDJ/BOLL/ATR/OBV、收益率与波动率、价差等（列名见函数注释）。

- engineer_features(df) ：实现 158 个特征的大头（K线形态、均线、回归、相关性、成交量等），大量用 TA-Lib/rolling 向量化。


In [4]:
import pandas as pd
import numpy as np
import joblib
import os
from tqdm import tqdm

# 特征工程
def _rolling_linear_regression(x, y):
    x = np.vstack([np.ones(len(x)), x]).T
    beta, res, _, _ = np.linalg.lstsq(x, y, rcond=None)
    return beta[1], res[0] if len(res) > 0 else 0.0, np.sum((y - (x @ beta))**2)
def engineer_features_158plus39(df):
    """
    计算39个技术指标特征和158个Alpha特征，并合并它们。
    """
    # 为了避免修改原始DataFrame，创建一个副本
    df_copy = df.copy()

    # 1. 计算158个Alpha特征
    df_158 = engineer_features(df_copy)
    
    # 2. 计算39个技术指标特征
    df_39 = engineer_features_39(df_copy)

    # 3. 合并两个DataFrame
    # 首先，从df_39中选取我们需要的列，避免与df_158中的原始列（如'开盘'）重复
    feature_cols_39 = [
        'sma_5', 'sma_20', 'ema_12', 'ema_26', 'rsi', 'macd', 'macd_signal', 
        'volume_change', 'obv', 'volume_ma_5', 'volume_ma_20', 'volume_ratio', 
        'kdj_k', 'kdj_d', 'kdj_j', 'boll_mid', 'boll_std', 'atr_14', 'ema_60', 
        'volatility_10', 'volatility_20', 'return_1', 'return_5', 'return_10',  
        'high_low_spread', 'open_close_spread', 'high_close_spread', 'low_close_spread'
    ]
    
    # 确保所有列都存在于df_39中
    feature_cols_39_exist = [col for col in feature_cols_39 if col in df_39.columns]
    
    # 合并，df_158 已经包含了原始列和158个特征
    df_final = pd.concat([df_158, df_39[feature_cols_39_exist]], axis=1)

    # 4. 处理可能因为合并产生的重复列（如果两个函数生成了同名特征）
    df_final = df_final.loc[:,~df_final.columns.duplicated()]

    # 5. 统一处理inf和NaN
    df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_final.fillna(0, inplace=True)
    
    return df_final

def engineer_features_39(df):
    """
    计算39个技术指标特征。
    'stock_idx','开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌额', '换手率', '涨跌幅',
    'sma_5', 'sma_20', 'ema_12', 'ema_26', 'rsi', 'macd', 'macd_signal', 'volume_change', 'obv',
    'volume_ma_5', 'volume_ma_20', 'volume_ratio', 'kdj_k', 'kdj_d', 'kdj_j', 'boll_mid', 'boll_std', 
    'atr_14', 'ema_60', 'volatility_10', 'volatility_20', 'return_1', 'return_5', 'return_10',  
    'high_low_spread', 'open_close_spread', 'high_close_spread', 'low_close_spread'
    """
    try:
        import talib
        import numpy as np
    except ImportError:
        print("请安装TA-Lib库: pip install TA-Lib")
        raise

    df = df.copy()

    # 基础变量
    open_ = df['开盘'].astype(float)
    high = df['最高'].astype(float)
    low = df['最低'].astype(float)
    close = df['收盘'].astype(float)
    volume = df['成交量'].astype(float)

    # 移动平均线 (SMA, EMA)
    df['sma_5'] = talib.SMA(close, timeperiod=5)
    df['sma_20'] = talib.SMA(close, timeperiod=20)
    df['ema_12'] = talib.EMA(close, timeperiod=12)
    df['ema_26'] = talib.EMA(close, timeperiod=26)
    df['ema_60'] = talib.EMA(close, timeperiod=60)

    # MACD
    macd_line, macd_signal_line, macd_hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    df['macd'] = macd_line
    df['macd_signal'] = macd_signal_line

    # RSI
    df['rsi'] = talib.RSI(close, timeperiod=14)

    # KDJ
    df['kdj_k'], df['kdj_d'] = talib.STOCH(high, low, close, fastk_period=9, slowk_period=3, slowd_period=3)
    df['kdj_j'] = 3 * df['kdj_k'] - 2 * df['kdj_d']

    # Bollinger Bands
    df['boll_mid'], df['boll_upper'], df['boll_lower'] = talib.BBANDS(close, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
    # 标准差 = (上轨 - 中轨) / 2
    df['boll_std'] = (df['boll_upper'] - df['boll_mid']) / 2

    # 删除临时列
    df.drop(columns=['boll_upper', 'boll_lower'], inplace=True)

    # ATR
    df['atr_14'] = talib.ATR(high, low, close, timeperiod=14)

    # OBV (On-Balance Volume)
    df['obv'] = talib.OBV(close, volume)

    # Volume-related features
    df['volume_change'] = volume.pct_change()
    df['volume_ma_5'] = talib.SMA(volume, timeperiod=5)
    df['volume_ma_20'] = talib.SMA(volume, timeperiod=20)
    df['volume_ratio'] = df['volume_ma_5'] / df['volume_ma_20']

    # Returns and Volatility
    df['return_1'] = close.pct_change(1)
    df['return_5'] = close.pct_change(5)
    df['return_10'] = close.pct_change(10)
    df['volatility_10'] = df['return_1'].rolling(10).std()
    df['volatility_20'] = df['return_1'].rolling(20).std()

    # Spreads
    df['high_low_spread'] = high - low
    df['open_close_spread'] = open_ - close
    df['high_close_spread'] = high - close
    df['low_close_spread'] = low - close

    # 处理 inf 和 -inf
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 填充 NaN 值（注意：这可能引入偏差，根据下游任务决定是否保留）
    df.fillna(0, inplace=True)

    return df


## 04数据集构建
#### 按“日期”组织一个样本，样本里是一堆股票的序列，每个股票的序列长度是 sequence_length。
- create_ranking_dataset_vectorized(...) 做的事：
- 把数据按 instrument(股票id) 和 datetime(日期) 排序。
- 对每只股票生成滑窗：窗口长度 sequence_length ，窗口结束日 end_date 对应一个 target=label[end_idx] （注意这里的 label 是在训练段里统一构造的）。
- 把所有窗口拼成 window_df(date, stock_code, seq, target) ，再按 date 聚合成“每日样本”：
  - day_seqs : (N, L, F)，N 是当日可用股票数
  - day_targets : (N,)
  - relevance : 根据 day_targets 排序分桶：top5=1.0，top20=0.5，top50=0.2，其余=0.0

In [5]:

def engineer_features(df):
    """
    使用talib加速特征计算
    """
    try:
        import talib
    except ImportError:
        print("请安装TA-Lib库: pip install TA-Lib")
        raise

    # 为了避免修改原始DataFrame，创建一个副本
    df = df.copy()

    # 基础变量
    open_ = df['开盘'].astype(float)
    high = df['最高'].astype(float)
    low = df['最低'].astype(float)
    close = df['收盘'].astype(float)
    volume = df['成交量'].astype(float)
    vwap = df['成交额'] / (volume + 1e-12)

    # 特征列表
    features = []
    feature_names = []

    # 1. K-line features (9 features) - 向量化操作，速度很快，无需更改
    features.extend([
        (close - open_) / (open_ + 1e-12),
        (high - low) / (open_ + 1e-12),
        (close - open_) / (high - low + 1e-12),
        (high - pd.concat([open_, close], axis=1).max(axis=1)) / (open_ + 1e-12),
        (high - pd.concat([open_, close], axis=1).max(axis=1)) / (high - low + 1e-12),
        (pd.concat([open_, close], axis=1).min(axis=1) - low) / (open_ + 1e-12),
        (pd.concat([open_, close], axis=1).min(axis=1) - low) / (high - low + 1e-12),
        (2 * close - high - low) / (open_ + 1e-12),
        (2 * close - high - low) / (high - low + 1e-12)
    ])
    feature_names.extend(['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2'])

    # 2. Price-related features (4 features) - 向量化操作，无需更改
    features.extend([
        open_ / (close + 1e-12),
        high / (close + 1e-12),
        low / (close + 1e-12),
        vwap / (close + 1e-12)
    ])
    feature_names.extend(['OPEN0', 'HIGH0', 'LOW0', 'VWAP0'])

    windows = [5, 10, 20, 30, 60]

    # 3. Price change features (5 features) - 向量化操作，无需更改
    for w in windows:
        features.append(close.shift(w) / (close + 1e-12))
        feature_names.append(f'ROC{w}')

    # 4. Moving average features (5 features) - 使用 talib 加速
    for w in windows:
        features.append(talib.SMA(close, timeperiod=w) / (close + 1e-12))
        feature_names.append(f'MA{w}')

    # 5. Standard deviation features (5 features) - 使用 talib 加速
    for w in windows:
        features.append(talib.STDDEV(close, timeperiod=w) / (close + 1e-12))
        feature_names.append(f'STD{w}')

    # 6. Regression-based features (15 features) - 使用 talib 加速
    for w in windows:
        slope = talib.LINEARREG_SLOPE(close, timeperiod=w)
        features.append(slope / (close + 1e-12))
        feature_names.append(f'BETA{w}')
        
        # R-squared can be calculated as CORREL^2
        time_period_series = pd.Series(range(w), index=close.index[:w])
        rolling_corr = close.rolling(w).corr(time_period_series)
        rsquare = rolling_corr**2
        features.append(rsquare)
        feature_names.append(f'RSQR{w}')

        # Residuals
        intercept = talib.LINEARREG_INTERCEPT(close, timeperiod=w)
        predicted = slope * (w - 1) + intercept
        resi = close - predicted
        features.append(resi / (close + 1e-12))
        feature_names.append(f'RESI{w}')

    # 7. Max/Min features (10 features) - 使用 talib 加速
    for w in windows:
        features.append(talib.MAX(high, timeperiod=w) / (close + 1e-12))
        feature_names.append(f'MAX{w}')
    for w in windows:
        features.append(talib.MIN(low, timeperiod=w) / (close + 1e-12))
        feature_names.append(f'MIN{w}')

    # 8. Quantile features (10 features) - talib 不支持，保留原实现
    for w in windows:
        features.append(close.rolling(w).quantile(0.8) / (close + 1e-12))
        feature_names.append(f'QTLU{w}')
    for w in windows:
        features.append(close.rolling(w).quantile(0.2) / (close + 1e-12))
        feature_names.append(f'QTLD{w}')

    # 9. Rank features (5 features) - talib 不支持，保留原实现
    for w in windows:
        features.append(close.rolling(w).rank(pct=True))
        feature_names.append(f'RANK{w}')

    # 10. Stochastic oscillator features (5 features) - talib.STOCH 计算的是另一指标，保留原实现
    for w in windows:
        min_low = low.rolling(w).min()
        max_high = high.rolling(w).max()
        features.append((close - min_low) / (max_high - min_low + 1e-12))
        feature_names.append(f'RSV{w}')

    # 11. Index of Max/Min features (15 features) - talib 不支持，保留原实现
    for w in windows:
        features.append(high.rolling(w).apply(np.argmax, raw=True) / w)
        feature_names.append(f'IMAX{w}')
    for w in windows:
        features.append(low.rolling(w).apply(np.argmin, raw=True) / w)
        feature_names.append(f'IMIN{w}')
    for w in windows:
        imax = high.rolling(w).apply(np.argmax, raw=True)
        imin = low.rolling(w).apply(np.argmin, raw=True)
        features.append((imax - imin) / w)
        feature_names.append(f'IMXD{w}')

    # 12. Correlation features (10 features) - 使用 talib 加速
    log_volume = np.log(volume + 1)
    for w in windows:
        features.append(talib.CORREL(close, log_volume, timeperiod=w))
        feature_names.append(f'CORR{w}')
    
    close_ret = close / close.shift(1)
    volume_ret = volume / (volume.shift(1) + 1e-12)
    log_volume_ret = np.log(volume_ret + 1)
    for w in windows:
        # talib.CORREL 需要 Series，且不能有 NaN
        corr_df = pd.concat([close_ret, log_volume_ret], axis=1).fillna(0)
        features.append(talib.CORREL(corr_df.iloc[:, 0], corr_df.iloc[:, 1], timeperiod=w))
        feature_names.append(f'CORD{w}')

    # 13. Count features (15 features) - 向量化操作，无需更改
    close_diff_pos = (close > close.shift(1))
    close_diff_neg = (close < close.shift(1))
    for w in windows:
        features.append(close_diff_pos.rolling(w).mean())
        feature_names.append(f'CNTP{w}')
    for w in windows:
        features.append(close_diff_neg.rolling(w).mean())
        feature_names.append(f'CNTN{w}')
    for w in windows:
        cntp = close_diff_pos.rolling(w).mean()
        cntn = close_diff_neg.rolling(w).mean()
        features.append(cntp - cntn)
        feature_names.append(f'CNTD{w}')

    # 14. Sum of price change features (15 features) - 向量化操作，无需更改
    close_diff_abs = (close - close.shift(1)).abs()
    close_diff_up = (close - close.shift(1)).clip(lower=0)
    close_diff_down = -(close - close.shift(1)).clip(upper=0)
    for w in windows:
        sum_abs = close_diff_abs.rolling(w).sum()
        sum_up = close_diff_up.rolling(w).sum()
        features.append(sum_up / (sum_abs + 1e-12))
        feature_names.append(f'SUMP{w}')
    for w in windows:
        sum_abs = close_diff_abs.rolling(w).sum()
        sum_down = close_diff_down.rolling(w).sum()
        features.append(sum_down / (sum_abs + 1e-12))
        feature_names.append(f'SUMN{w}')
    for w in windows:
        sum_abs = close_diff_abs.rolling(w).sum()
        sum_up = close_diff_up.rolling(w).sum()
        sum_down = close_diff_down.rolling(w).sum()
        features.append((sum_up - sum_down) / (sum_abs + 1e-12))
        feature_names.append(f'SUMD{w}')

    # 15. Volume-related features (10 features) - 使用 talib 加速
    for w in windows:
        features.append(talib.SMA(volume, timeperiod=w) / (volume + 1e-12))
        feature_names.append(f'VMA{w}')
    for w in windows:
        features.append(talib.STDDEV(volume, timeperiod=w) / (volume + 1e-12))
        feature_names.append(f'VSTD{w}')

    # 16. Weighted volume features (5 features) - 向量化操作，无需更改
    vol_weighted_ret = (close / close.shift(1) - 1).abs() * volume
    for w in windows:
        mean_vol_w_ret = vol_weighted_ret.rolling(w).mean()
        std_vol_w_ret = vol_weighted_ret.rolling(w).std()
        features.append(std_vol_w_ret / (mean_vol_w_ret + 1e-12))
        feature_names.append(f'WVMA{w}')

    # 17. Volume change sum features (15 features) - 向量化操作，无需更改
    volume_diff_abs = (volume - volume.shift(1)).abs()
    volume_diff_up = (volume - volume.shift(1)).clip(lower=0)
    volume_diff_down = -(volume - volume.shift(1)).clip(upper=0)
    for w in windows:
        sum_abs = volume_diff_abs.rolling(w).sum()
        sum_up = volume_diff_up.rolling(w).sum()
        features.append(sum_up / (sum_abs + 1e-12))
        feature_names.append(f'VSUMP{w}')
    for w in windows:
        sum_abs = volume_diff_abs.rolling(w).sum()
        sum_down = volume_diff_down.rolling(w).sum()
        features.append(sum_down / (sum_abs + 1e-12))
        feature_names.append(f'VSUMN{w}')
    for w in windows:
        sum_abs = volume_diff_abs.rolling(w).sum()
        sum_up = volume_diff_up.rolling(w).sum()
        sum_down = volume_diff_down.rolling(w).sum()
        features.append((sum_up - sum_down) / (sum_abs + 1e-12))
        feature_names.append(f'VSUMD{w}')

    # Combine all features into a new DataFrame
    feature_df = pd.concat(features, axis=1)
    feature_df.columns = feature_names
    
    # Merge with original df
    df = pd.concat([df, feature_df], axis=1)
    
    # 填充缺失值
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df
def process_single_stock(stock_row, data, features, sequence_length, date):
    """处理单只股票的数据，返回序列、目标值和股票索引"""
    stock_code = stock_row['instrument']
    # stock_idx = stock_row['stock_idx']
    
    # 获取该股票历史sequence_length天的数据（包括当天）
    stock_history = data[
        (data['instrument'] == stock_code) & 
        (data['datetime'] <= date)
    ].sort_values('datetime').tail(sequence_length)

    if len(stock_history) == sequence_length:
        seq = stock_history[features].values
        target = stock_row['label']  # 下一天的涨跌幅
        return seq, target, stock_code
    else:
        return None, None, None

def process_single_date(date, data, features, sequence_length):
    """处理单个日期的所有股票数据"""
    try:
        # 获取当天有target的股票（即有下一天数据的股票）
        day_data = data[data['datetime'] == date]
        day_data = day_data.dropna(subset=['label'])  # 确保有target
        
        if len(day_data) < 10:  # 确保至少有10只股票
            return None
            
        # 获取当天所有股票的特征序列
        day_sequences = []
        day_targets = []
        day_stock_indices = []
        
        # 对于单个日期内的股票处理，仍使用串行方式避免过度并行化
        # 因为多进程的开销可能超过收益
        for _, stock_row in day_data.iterrows():
            seq, target, stock_idx = process_single_stock(
                stock_row, data, features, sequence_length, date
            )
            if seq is not None:
                day_sequences.append(seq)
                day_targets.append(target)
                day_stock_indices.append(stock_idx)
        
        if len(day_sequences) >= 10:  # 确保有足够的股票
            # 创建排序标签：涨跌幅越高，相关性得分越高
            day_targets = np.array(day_targets)
            # 使用涨跌幅的排序作为相关性得分（值越大排名越高）
            sorted_indices = np.argsort(day_targets)[::-1]  # 降序排列
            relevance = np.zeros_like(day_targets, dtype=np.float32)

            for rank, idx in enumerate(sorted_indices):
                if rank < 5:
                    relevance[idx] = 1.0
                elif rank < 20:
                    relevance[idx] = 0.5
                elif rank < 50:
                    relevance[idx] = 0.2
                else:
                    relevance[idx] = 0.0

            # for rank, idx in enumerate(sorted_indices):
            #     relevance[idx] = len(day_targets) - rank  # 最高涨跌幅得分最高
            
            return {
                'sequences': np.array(day_sequences),
                'targets': day_targets,
                'relevance': relevance,
                'stock_indices': day_stock_indices,
                'date': date
            }
        else:
            return None
            
    except Exception as e:
        print(f"处理日期 {date} 时出错: {e}")
        return None

def create_ranking_dataset_multiprocess(data, features, sequence_length, ranking_data_path=None, max_workers=None):
    """
    输入：股票历史数据 DataFrame，特征列名列表，序列长度，排名数据保存路径，最大工作进程数
    输出：排序数据集，格式为：(sequences, targets, relevance_scores, stock_indices)
    - sequences: List of np.array, 每个元素形状为 (num_stocks, sequence_length, num_features)
    - targets: List of np.array, 每个元素形状为 (num_stocks,)
    - relevance_scores: List of np.array, 每个元素形状为 (num_stocks,)
    - stock_indices: List of List, 每个元素为对应股票的索引列表
    """
    """多进程版本的排序数据集创建函数"""
    if ranking_data_path is not None:
        # 如果指定了ranking_data_path，尝试加载已有的数据集
        if os.path.exists(ranking_data_path):
            print(f"加载已有的排序数据集: {ranking_data_path}")
            return joblib.load(ranking_data_path)
    """
    创建排序数据集，按日期组织数据，每个样本包含同一天所有股票的特征和涨跌幅排序
    使用多线程加速处理
    """
    sequences = []
    targets = []
    relevance_scores = []
    stock_indices = []
    
    print("正在创建排序数据集（多线程版本）...")
    
    # 获取所有日期，确保有足够的历史数据
    all_dates = sorted(data['datetime'].unique())
    min_date_for_sequences = all_dates[sequence_length-1]  # 确保有足够历史数据
    
    # 只使用有足够历史数据的日期
    valid_dates = [date for date in all_dates if date >= min_date_for_sequences]
    
    print(f"总日期数: {len(all_dates)}, 有效日期数: {len(valid_dates)}")
    
    # 设置最大工作进程数
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor
    from functools import partial
    from tqdm import tqdm
    if max_workers is None:
        max_workers = min(mp.cpu_count(), 10)
    
    print(f"使用 {max_workers} 个进程处理数据")
    
    # 分批处理日期以避免内存问题
    processed_count = 0
        
    # 使用进程池并行处理日期批次
    try:
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # 创建处理函数的偏函数
            process_func = partial(process_single_date,
                                    data=data,
                                    features=features,
                                    sequence_length=sequence_length)
            
            # 并行处理批次中的所有日期
            futures = [executor.submit(process_func, date) for date in valid_dates]
            
            # 收集结果
            for future in tqdm(futures, desc="Processing dates", total=len(valid_dates)):
                try:
                    result = future.result(timeout=60)  # 设置超时
                    if result is not None:
                        sequences.append(result['sequences'])
                        targets.append(result['targets'])
                        relevance_scores.append(result['relevance'])
                        stock_indices.append(result['stock_indices'])
                        processed_count += 1
                except Exception as e:
                    print(f"处理某个日期时出错: {e}")
                    continue
                    
    except Exception as e:
        print(f"进程池处理出错，回退到串行处理: {e}")
        # 如果多进程出错，回退到串行处理
        for date in tqdm(valid_dates, desc="串行处理"):
            result = process_single_date(date, data, features, sequence_length)
            if result is not None:
                sequences.append(result['sequences'])
                targets.append(result['targets'])
                relevance_scores.append(result['relevance'])
                stock_indices.append(result['stock_indices'])
                processed_count += 1
    
    print(f"成功创建 {len(sequences)} 个训练样本")
    if len(sequences) > 0:
        print(f"每个样本平均包含 {np.mean([len(seq) for seq in sequences]):.1f} 只股票")
    
    # 将四个数据保存下来，下次直接读取
    if ranking_data_path:
        joblib.dump((sequences, targets, relevance_scores, stock_indices), ranking_data_path)
        print(f"数据集已保存到: {ranking_data_path}")
    
    return sequences, targets, relevance_scores, stock_indices

def create_dataset(data, features, sequence_length, ranking_data_path=None):
    """保持原有接口，但内部调用新的排序数据集创建函数"""
    return create_ranking_dataset_multiprocess(data, features, sequence_length, ranking_data_path)

def create_ranking_dataset_vectorized(data, features, sequence_length, ranking_data_path=None, min_window_end_date=None):
    """
    向量化加速版本：预计算每只股票的所有滑动窗口，再按日期聚合。
    保持与原函数完全相同的输出格式。
    """
    # if ranking_data_path and os.path.exists(ranking_data_path):
    #     print(f"加载已有的排序数据集: {ranking_data_path}")
    #     return joblib.load(ranking_data_path)

    print("正在创建排序数据集（向量化加速版本）...")
    # data.rename(columns={'stock_idx': 'instrument'}, inplace=True)
    data = data.copy()
    data.rename(columns={'日期': 'datetime'}, inplace=True)
    data['datetime'] = pd.to_datetime(data['datetime'])

    # 1. 确保数据按股票和时间排序
    data = data.sort_values(['instrument', 'datetime']).reset_index(drop=True)
    
    # 2. 确保每只股票都有 'label'（次日涨跌幅），否则无法作为 target
    data = data.dropna(subset=['label'])
    
    # 3. 为每只股票生成所有滑动窗口
    # 仅保留满足以下条件的 end_date：
    # - 历史窗口长度满足 sequence_length
    # - end_date 之后存在 5 条未来数据
    # - 这 5 条未来数据在自然日上连续（任意节假日/周末导致的日期跳跃都会被过滤）
    all_windows = []  # 每个元素: (end_date, stock_code, sequence, target)

    print("Step 1: 为每只股票生成滑动窗口...")
    grouped = data.groupby('instrument')
    
    for stock_code, group in tqdm(grouped, desc="Processing stocks"):
        if len(group) < sequence_length:
            continue
        
        # 提取特征和 label
        feature_values = group[features].values.astype(np.float32)  # (T, F)
        labels = group['label'].values.astype(np.float32)           # (T,)
        dates = group['datetime'].values                            # (T,)
        dates_day = group['datetime'].values.astype('datetime64[D]')

        # 生成滑动窗口：从第 sequence_length-1 行开始（0-indexed）
        num_windows = len(group) - sequence_length + 1
        n = len(group)
        for i in range(num_windows):
            end_idx = i + sequence_length - 1

            # 需要有未来 5 条数据
            if end_idx + 5 >= n:
                continue

            # # 未来 5 条数据日期必须连续（自然日相邻）
            # future_dates = dates_day[end_idx + 1:end_idx + 6]
            # future_diffs = np.diff(future_dates).astype(np.int64)
            # if not np.all(future_diffs == 1):
            #     continue

            seq = feature_values[i : i + sequence_length]   # (L, F)
            target = labels[end_idx]                        # label 对应窗口最后一天的次日涨跌幅
            end_date = dates[end_idx]                       # 窗口结束日期（即预测日）

            if not np.isfinite(target):
                continue        

            all_windows.append((end_date, stock_code, seq, target))

    # 4. 转为 DataFrame 便于按日期聚合
    print("Step 2: 按日期聚合窗口...")
    window_df = pd.DataFrame(all_windows, columns=['date', 'stock_code', 'seq', 'target'])

    # 5. 按 date 分组，构建每日样本
    sequences = []
    targets = []
    relevance_scores = []
    stock_indices = []

    print("Step 3: 构建每日样本并计算 relevance...")
    grouped_by_date = window_df.groupby('date')

    if min_window_end_date is not None:
        min_window_end_date = pd.to_datetime(min_window_end_date)
    
    for date, group in tqdm(grouped_by_date, desc="Aggregating by date"):
        if min_window_end_date is not None and pd.to_datetime(date) < min_window_end_date:
            continue

        if len(group) < 10:
            continue
        
        # 提取数据
        day_seqs = np.stack(group['seq'].values)          # (N, L, F)
        day_targets = group['target'].values              # (N,)
        day_stocks = group['stock_code'].tolist()         # [str]

        # 计算 relevance（与原逻辑一致）
        sorted_indices = np.argsort(day_targets)[::-1]
        relevance = np.zeros_like(day_targets, dtype=np.float32)
        # for rank, idx in enumerate(sorted_indices):
        #     relevance[idx] = len(day_targets) - rank

        for rank, idx in enumerate(sorted_indices):
            if rank < 5:
                relevance[idx] = 1.0
            elif rank < 20:
                relevance[idx] = 0.5
            elif rank < 50:
                relevance[idx] = 0.2
            else:
                relevance[idx] = 0.0

        sequences.append(day_seqs)
        targets.append(day_targets)
        relevance_scores.append(relevance)
        stock_indices.append(day_stocks)

    print(f"成功创建 {len(sequences)} 个训练样本")
    if len(sequences) > 0:
        avg_stocks = np.mean([len(seq) for seq in sequences])
        print(f"每个样本平均包含 {avg_stocks:.1f} 只股票")


    return sequences, targets, relevance_scores, stock_indices

## 05模型训练
#### 全流程入口：读数据→特征→标准化→建样本→训练→early stop→保存 best_model


#### 标签构造
- open_t1 = 开盘.shift(-1) ， open_t5 = 开盘.shift(-5)
- label = (open_t5 - open_t1) / open_t1
 也就是“从下一天开盘到第5天开盘”的 5 日收益率（并过滤极小开盘价避免爆炸）。

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from tensorboardX import SummaryWriter
import joblib
import os
import json
import multiprocessing as mp
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

feature_cloums_map = {
    '39': ['开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌额', '换手率', '涨跌幅','sma_5', 'sma_20', 'ema_12', 'ema_26', 'rsi', 'macd', 'macd_signal', 'volume_change', 'obv','volume_ma_5', 'volume_ma_20', 'volume_ratio', 'kdj_k', 'kdj_d', 'kdj_j', 'boll_mid', 'boll_std', 'atr_14', 'ema_60', 'volatility_10', 'volatility_20', 'return_1', 'return_5', 'return_10',  'high_low_spread', 'open_close_spread', 'high_close_spread', 'low_close_spread'],

    '158+39': ['开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌额', '换手率', '涨跌幅','KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2', 'OPEN0', 'HIGH0', 'LOW0', 'VWAP0', 'ROC5', 'ROC10', 'ROC20', 'ROC30', 'ROC60', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60', 'STD5', 'STD10', 'STD20', 'STD30', 'STD60', 'BETA5', 'BETA10', 'BETA20', 'BETA30', 'BETA60', 'RSQR5', 'RSQR10', 'RSQR20', 'RSQR30', 'RSQR60', 'RESI5', 'RESI10', 'RESI20', 'RESI30', 'RESI60', 'MAX5', 'MAX10', 'MAX20', 'MAX30', 'MAX60', 'MIN5', 'MIN10', 'MIN20', 'MIN30', 'MIN60', 'QTLU5', 'QTLU10', 'QTLU20', 'QTLU30', 'QTLU60', 'QTLD5', 'QTLD10', 'QTLD20', 'QTLD30', 'QTLD60', 'RANK5', 'RANK10', 'RANK20', 'RANK30', 'RANK60', 'RSV5', 'RSV10', 'RSV20', 'RSV30', 'RSV60', 'IMAX5', 'IMAX10', 'IMAX20', 'IMAX30', 'IMAX60', 'IMIN5', 'IMIN10', 'IMIN20', 'IMIN30', 'IMIN60', 'IMXD5', 'IMXD10', 'IMXD20', 'IMXD30', 'IMXD60', 'CORR5', 'CORR10', 'CORR20', 'CORR30', 'CORR60', 'CORD5', 'CORD10', 'CORD20', 'CORD30', 'CORD60', 'CNTP5', 'CNTP10', 'CNTP20', 'CNTP30', 'CNTP60', 'CNTN5', 'CNTN10', 'CNTN20', 'CNTN30', 'CNTN60', 'CNTD5', 'CNTD10', 'CNTD20', 'CNTD30', 'CNTD60', 'SUMP5', 'SUMP10', 'SUMP20', 'SUMP30', 'SUMP60', 'SUMN5', 'SUMN10', 'SUMN20', 'SUMN30', 'SUMN60', 'SUMD5', 'SUMD10', 'SUMD20', 'SUMD30', 'SUMD60', 'VMA5', 'VMA10', 'VMA20', 'VMA30', 'VMA60', 'VSTD5', 'VSTD10', 'VSTD20', 'VSTD30', 'VSTD60', 'WVMA5', 'WVMA10', 'WVMA20', 'WVMA30', 'WVMA60', 'VSUMP5', 'VSUMP10', 'VSUMP20', 'VSUMP30', 'VSUMP60', 'VSUMN5', 'VSUMN10', 'VSUMN20', 'VSUMN30', 'VSUMN60', 'VSUMD5', 'VSUMD10', 'VSUMD20', 'VSUMD30', 'VSUMD60','sma_5', 'sma_20', 'ema_12', 'ema_26', 'rsi', 'macd', 'macd_signal', 'volume_change', 'obv', 'volume_ma_5', 'volume_ma_20', 'volume_ratio', 'kdj_k', 'kdj_d', 'kdj_j', 'boll_mid', 'boll_std', 'atr_14', 'ema_60', 'volatility_10', 'volatility_20', 'return_1', 'return_5', 'return_10',  'high_low_spread', 'open_close_spread', 'high_close_spread', 'low_close_spread']
}
feature_engineer_func_map = {
    '39': engineer_features_39,
    '158+39': engineer_features_158plus39
}



#### 预处理
- 对每只股票做特征工程（此处为了 notebook 稳定性用串行 tqdm）
- 映射 instrument = 股票代码 -> idx
- 构造 label 并清洗

In [7]:

def _build_label_and_clean(processed, drop_small_open=True):
    """统一构建标签并清洗无效样本。"""
    processed['open_t1'] = processed.groupby('股票代码')['开盘'].shift(-1)
    processed['open_t5'] = processed.groupby('股票代码')['开盘'].shift(-5)

    # 过滤无效开盘价，避免收益率极端爆炸
    if drop_small_open:
        processed = processed[processed['open_t1'] > 1e-4]

    processed['label'] = (processed['open_t5'] - processed['open_t1']) / (processed['open_t1'] + 1e-12)
    processed = processed.dropna(subset=['label'])

    processed.drop(columns=['open_t1', 'open_t5'], inplace=True)
    return processed


# def _preprocess_common(df, stockid2idx, desc, drop_small_open=True):
#     assert config['feature_num'] in feature_engineer_func_map, f"Unsupported feature_num: {config['feature_num']}"
#     assert stockid2idx is not None, "stockid2idx 不能为空"
#     feature_engineer = feature_engineer_func_map[config['feature_num']]
#     feature_columns = feature_cloums_map[config['feature_num']]

#     # 保证时序正确，避免 shift 标签错位
#     df = df.copy()
#     df = df.sort_values(['股票代码', '日期']).reset_index(drop=True)

#     print(f"正在使用多进程进行{desc}...")
#     groups = [group for _, group in df.groupby('股票代码', sort=False)]
#     if len(groups) == 0:
#         raise ValueError(f"{desc}输入为空，无法继续")

#     num_processes = min(10, mp.cpu_count())
#     with mp.Pool(processes=num_processes) as pool:
#         processed_list = list(tqdm(pool.imap(feature_engineer, groups), total=len(groups), desc=desc))

#     processed = pd.concat(processed_list).reset_index(drop=True)

#     # 映射股票索引，并剔除映射失败样本
#     processed['instrument'] = processed['股票代码'].map(stockid2idx)
#     processed = processed.dropna(subset=['instrument']).copy()
#     processed['instrument'] = processed['instrument'].astype(np.int64)

#     processed = _build_label_and_clean(processed, drop_small_open=drop_small_open)
#     return processed, feature_columns
#--------------------------------------------数据预处理封装-----------------------------------------
def _preprocess_common(df, stockid2idx, desc, drop_small_open=True):
    assert config['feature_num'] in feature_engineer_func_map, f"Unsupported feature_num: {config['feature_num']}"
    assert stockid2idx is not None, "stockid2idx 不能为空"

    feature_engineer = feature_engineer_func_map[config['feature_num']]
    feature_columns = feature_cloums_map[config['feature_num']]

    df = df.copy()
    df['股票代码'] = df['股票代码'].astype(str).str.zfill(6)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.sort_values(['股票代码', '日期']).reset_index(drop=True)

    print(f"正在进行{desc}，当前使用串行模式，避免 Notebook 多进程卡死...")

    groups = [group for _, group in df.groupby('股票代码', sort=False)]
    if len(groups) == 0:
        raise ValueError(f"{desc}输入为空，无法继续")

    processed_list = []
    for group in tqdm(groups, total=len(groups), desc=desc):
        processed_list.append(feature_engineer(group))

    processed = pd.concat(processed_list).reset_index(drop=True)

    processed['instrument'] = processed['股票代码'].map(stockid2idx)
    processed = processed.dropna(subset=['instrument']).copy()
    processed['instrument'] = processed['instrument'].astype(np.int64)

    processed = _build_label_and_clean(processed, drop_small_open=drop_small_open)

    return processed, feature_columns


# 数据预处理函数
def preprocess_data(df, is_train=True, stockid2idx=None):
    if not is_train:
        return _preprocess_common(df, stockid2idx, desc="特征工程", drop_small_open=False)
    return _preprocess_common(df, stockid2idx, desc="特征工程", drop_small_open=True)


def preprocess_val_data(df, stockid2idx=None):
    # 验证集与训练集保持同口径，避免 label 分布漂移
    return _preprocess_common(df, stockid2idx, desc="验证集特征工程", drop_small_open=True)



#### 损失函数
- `WeightedRankingLoss`：listwise 部分对 y_pred 与 y_true 做 softmax 后计算加权交叉熵，对真实 Top-5 样本加大权重（`top5_weight=3`）
- pairwise 部分为两两比较的 sigmoid loss，权重由 `pairwise_weight` 控制（本方案取 0，即只用 listwise）
- 优化器 AdamW，学习率 3e-5，权重衰减 1e-5；学习率按 `LinearLR` 从 1.0 线性降到 0.2

In [8]:

# 加权的排序损失函数
class WeightedRankingLoss(nn.Module):
    """
    组合的加权排序损失函数，着重强调top-k的样本。
    """
    def __init__(self, temperature=1.0, k=5, weight_factor=2.0, pairwise_weight=1, base_weight=1.0):
        super(WeightedRankingLoss, self).__init__()
        self.temperature = temperature
        self.k = k
        self.weight_factor = weight_factor
        self.pairwise_weight = pairwise_weight
        self.base_weight = base_weight

    def listwise_loss(self, y_pred, y_true, weights):
        """加权的Listwise损失 (KL散度 + Cross Entropy)"""
        
        pred_probs = F.softmax(y_pred / self.temperature, dim=1)
        target_probs = F.softmax(y_true / self.temperature, dim=1)

        # 加权 Cross Entropy（原实现未使用 weights）
        weighted_ce = -(target_probs * torch.log(pred_probs + 1e-12) * weights)
        ce_loss = (weighted_ce.sum(dim=1) / (weights.sum(dim=1) + 1e-12)).mean()
        
        return ce_loss

    def pairwise_loss(self, y_pred, y_true, weights):
        """加权的Pairwise损失"""
        batch_size, num_items = y_pred.size()
        
        pred_diff = y_pred.unsqueeze(2) - y_pred.unsqueeze(1)
        true_diff = y_true.unsqueeze(2) - y_true.unsqueeze(1)
        
        # 只考虑真实标签不同的项目对
        mask = (true_diff != 0).float()
        
        # 创建权重矩阵
        # 如果一对(i, j)中，i或j是关键样本，则权重更高
        weight_matrix = weights.unsqueeze(2) + weights.unsqueeze(1)
        # weight_matrix = torch.where(weight_matrix > 2.0, self.weight_factor, 1.0)
        
        pairwise_loss = torch.sigmoid(-pred_diff * torch.sign(true_diff))
        
        # 应用mask和权重
        weighted_loss = pairwise_loss * mask * weight_matrix
        
        num_pairs = mask.sum(dim=[1, 2]).clamp(min=1)
        loss = (weighted_loss.sum(dim=[1, 2]) / num_pairs).mean()
        
        return loss
        
    def forward(self, y_pred, y_true):
        """
        y_pred: [batch, num_items]
        y_true: [batch, num_items] (真实涨跌幅)
        """
        batch_size, num_items = y_true.size()
        k = min(self.k, num_items)

        # 1. 识别 top-k 的样本
        _, top_indices = torch.topk(y_true, k, dim=1)
        
        # 2. 创建权重向量
        weights = torch.full_like(y_true, fill_value=self.base_weight)
        for i in range(batch_size):
            weights[i, top_indices[i]] = self.weight_factor
            
        # 3. 计算加权损失
        listwise = self.listwise_loss(y_pred, y_true, weights)
        pairwise = self.pairwise_loss(y_pred, y_true, weights)
        
        # 组合两种损失
        total_loss = listwise + self.pairwise_weight * pairwise
        
        return total_loss


#### 评估指标（final_score）
- 用模型分数选 top5，看真实收益率之和（pred_return_sum）
- 计算理论最优 top5（max_return_sum）和随机期望（random_return_sum = 5 * mean）
- final_score = (pred - random) / (max - random) （做了数值稳定处理）

In [9]:

def calculate_ranking_metrics(y_pred, y_true, masks, k=5):
    """计算新的评估指标：Top 5 收益之和，以及与理论最高值和随机值的比值"""
    batch_size = y_pred.size(0)
    
    # Metrics accumulators
    pred_return_sum_list = []
    max_return_sum_list = []
    random_return_sum_list = []
    ratio_pred_list = []
    ratio_random_list = []
    final_score_list = []
    
    for i in range(batch_size):
        mask = masks[i]
        valid_indices = mask.nonzero().squeeze()
        
        if valid_indices.numel() < k:
            continue
            
        valid_pred = y_pred[i][valid_indices]
        valid_true = y_true[i][valid_indices] # This is the 5-day return
        
        # 1. Predicted Top 5
        _, pred_indices = torch.topk(valid_pred, k)
        pred_top_returns = valid_true[pred_indices]
        pred_return_sum = pred_top_returns.sum().item()
        
        # 2. True Top 5 (Theoretical Max)
        _, true_indices = torch.topk(valid_true, k)
        true_top_returns = valid_true[true_indices]
        max_return_sum = true_top_returns.sum().item()
        
        # 3. Random 5 (Expected Value)
        # Expected sum = 5 * mean(all valid returns)
        random_return_sum = k * valid_true.mean().item()
        
        # 计算每个样本的比例与稳定化 final_score
        ratio_pred = pred_return_sum / (max_return_sum + 1e-12) if abs(max_return_sum) > 1e-9 else 0.0
        ratio_random = random_return_sum / (max_return_sum + 1e-12) if abs(max_return_sum) > 1e-9 else 0.0
        denominator = max_return_sum - random_return_sum
        final_score = (pred_return_sum - random_return_sum) / (denominator + 1e-12) if abs(denominator) > 1e-6 else 0.0
        
        pred_return_sum_list.append(pred_return_sum)
        max_return_sum_list.append(max_return_sum)
        random_return_sum_list.append(random_return_sum)
        ratio_pred_list.append(ratio_pred)
        ratio_random_list.append(ratio_random)
        final_score_list.append(final_score)
        
    metrics = {
        'pred_return_sum': np.mean(pred_return_sum_list) if pred_return_sum_list else 0.0,
        'max_return_sum': np.mean(max_return_sum_list) if max_return_sum_list else 0.0,
        'random_return_sum': np.mean(random_return_sum_list) if random_return_sum_list else 0.0,
    }
    
    # 比值用逐样本均值，降低极端日影响
    metrics['ratio_pred'] = np.mean(ratio_pred_list) if ratio_pred_list else 0.0
    metrics['ratio_random'] = np.mean(ratio_random_list) if ratio_random_list else 0.0
    metrics['final_score'] = np.mean(final_score_list) if final_score_list else 0.0
    
    return metrics

class RankingDataset(torch.utils.data.Dataset):
    """排序数据集类"""
    def __init__(self, sequences, targets, relevance_scores, stock_indices):
        self.sequences = sequences
        self.targets = targets
        self.relevance_scores = relevance_scores
        self.stock_indices = stock_indices
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return {
            'sequences': torch.FloatTensor(self.sequences[idx]),  # [num_stocks, seq_len, features]
            'targets': torch.FloatTensor(self.targets[idx]),      # [num_stocks] 真实涨跌幅
            'relevance': torch.LongTensor(self.relevance_scores[idx]),  # [num_stocks] 排序标签
            'stock_indices': torch.LongTensor(self.stock_indices[idx])  # [num_stocks] 股票索引
        }

def collate_fn(batch):
    """自定义collate函数处理变长序列"""
    sequences = [item['sequences'] for item in batch]
    targets = [item['targets'] for item in batch]
    relevance = [item['relevance'] for item in batch]
    stock_indices = [item['stock_indices'] for item in batch]
    
    # 找到最大股票数量
    max_stocks = max(seq.size(0) for seq in sequences)
    
    # Padding到相同长度
    padded_sequences = []
    padded_targets = []
    padded_relevance = []
    padded_stock_indices = []
    masks = []
    
    for seq, tgt, rel, stock_idx in zip(sequences, targets, relevance, stock_indices):
        num_stocks = seq.size(0)
        seq_len = seq.size(1)
        feature_dim = seq.size(2)
        
        # 创建padding
        if num_stocks < max_stocks:
            pad_size = max_stocks - num_stocks
            seq_pad = torch.zeros(pad_size, seq_len, feature_dim)
            tgt_pad = torch.zeros(pad_size)
            rel_pad = torch.zeros(pad_size, dtype=torch.long)
            stock_pad = torch.zeros(pad_size, dtype=torch.long)
            
            seq = torch.cat([seq, seq_pad], dim=0)
            tgt = torch.cat([tgt, tgt_pad], dim=0)
            rel = torch.cat([rel, rel_pad], dim=0)
            stock_idx = torch.cat([stock_idx, stock_pad], dim=0)
        
        # 创建mask标记有效位置
        mask = torch.ones(max_stocks)
        mask[num_stocks:] = 0
        
        padded_sequences.append(seq)
        padded_targets.append(tgt)
        padded_relevance.append(rel)
        padded_stock_indices.append(stock_idx)
        masks.append(mask)
    
    return {
        'sequences': torch.stack(padded_sequences),      # [batch, max_stocks, seq_len, features]
        'targets': torch.stack(padded_targets),          # [batch, max_stocks]
        'relevance': torch.stack(padded_relevance),      # [batch, max_stocks]
        'stock_indices': torch.stack(padded_stock_indices),  # [batch, max_stocks]
        'masks': torch.stack(masks)                      # [batch, max_stocks]
    }


#### 训练/验证循环
- train 用预处理阶段算好的 relevance （分桶的 1/0.5/0.2/0）来喂给 criterion
- eval 里则临时用 valid_true 的排序生成 relevance_scores = [N..1] 再算损失（这点和 train 的 relevance 定义不完全一致）

In [10]:

# 排序训练函数
def train_ranking_model(model, dataloader, criterion, optimizer, device, epoch, writer):
    model.train()
    total_loss = 0
    total_metrics = {}
    local_step = 0
    
    for batch in tqdm(dataloader, desc=f"Training Epoch {epoch+1}"):
        sequences = batch['sequences'].to(device)    # [batch, max_stocks, seq_len, features]
        targets = batch['targets'].to(device)        # [batch, max_stocks] 真实涨跌幅
        relevance = batch['relevance'].to(device)    # [batch, max_stocks] 预处理的相关性得分
        masks = batch['masks'].to(device)            # [batch, max_stocks] 有效位置mask
        
        optimizer.zero_grad()
        
        # 模型预测
        outputs = model(sequences, masks)  # [batch, max_stocks] 预测分数
        
        # 应用mask，只考虑有效股票
        masked_outputs = outputs * masks + (1 - masks) * (-1e9)  # 无效位置设为很小的值
        masked_targets = targets * masks
        masked_relevance = relevance.float() * masks  # 使用预处理好的相关性得分
        
        # 计算损失（只对有效股票计算）
        batch_loss = None
        batch_size = sequences.size(0)
        
        for i in range(batch_size):
            mask = masks[i]
            valid_indices = mask.nonzero().squeeze()
            
            if valid_indices.numel() == 0:
                continue
                
            if valid_indices.dim() == 0:
                valid_indices = valid_indices.unsqueeze(0)
            
            # 获取有效股票的预测值和预处理好的相关性得分
            valid_pred = masked_outputs[i][valid_indices]
            valid_relevance = masked_relevance[i][valid_indices]
            
            if len(valid_pred) > 1:
                # 直接使用预处理好的相关性得分，无需重新计算
                loss = criterion(valid_pred.unsqueeze(0), valid_relevance.unsqueeze(0))
                batch_loss = batch_loss + loss if isinstance(batch_loss, torch.Tensor) else loss
        
        if batch_loss is not None:
            batch_loss = batch_loss / batch_size
            batch_loss.backward()
            if not config.get('drop_clip', True):
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                if writer:
                    writer.add_scalar('train/grad_norm', grad_norm, global_step=epoch*len(dataloader)+local_step)
            optimizer.step()
            
            total_loss += batch_loss.item()
            
            # 计算评估指标
            with torch.no_grad():
                metrics = calculate_ranking_metrics(masked_outputs, masked_targets, masks, k=5)
                for k, v in metrics.items():
                    if k not in total_metrics:
                        total_metrics[k] = 0
                    total_metrics[k] += v
            
            local_step += 1
            if writer:
                writer.add_scalar('train/loss', batch_loss.item(), global_step=epoch*len(dataloader)+local_step)
                for k, v in metrics.items():
                    writer.add_scalar(f'train/{k}', v, global_step=epoch*len(dataloader)+local_step)
    
    # 计算平均指标
    if local_step > 0:
        for k in total_metrics:
            total_metrics[k] /= local_step
    
    return total_loss / len(dataloader) if len(dataloader) > 0 else 0, total_metrics

def evaluate_ranking_model(model, dataloader, criterion, device, writer, epoch):
    model.eval()
    total_loss = 0
    total_metrics = {}
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating Epoch {epoch+1}"):
            sequences = batch['sequences'].to(device)
            targets = batch['targets'].to(device)
            masks = batch['masks'].to(device)
            
            # 模型预测
            outputs = model(sequences,masks)
            
            # 应用mask
            masked_outputs = outputs * masks + (1 - masks) * (-1e9)
            masked_targets = targets * masks
            
            # 计算损失
            batch_loss = None
            batch_size = sequences.size(0)
            
            for i in range(batch_size):
                mask = masks[i]
                valid_indices = mask.nonzero().squeeze()
                
                if valid_indices.numel() == 0:
                    continue
                    
                if valid_indices.dim() == 0:
                    valid_indices = valid_indices.unsqueeze(0)
                
                valid_pred = masked_outputs[i][valid_indices]
                valid_true = masked_targets[i][valid_indices]
                
                if len(valid_pred) > 1:
                    _, sorted_indices = torch.sort(valid_true, descending=True)
                    relevance_scores = torch.zeros_like(valid_true, requires_grad=False)
                    relevance_scores[sorted_indices] = torch.arange(len(valid_true), 0, -1, device=device, dtype=torch.float32)
                    relevance_scores = relevance_scores.detach()
                    
                    loss = criterion(valid_pred.unsqueeze(0), relevance_scores.unsqueeze(0))
                    batch_loss = batch_loss + loss if batch_loss is not None else loss
            
            if batch_loss is not None:
                batch_loss = batch_loss / batch_size
                total_loss += batch_loss.item()
            
            # 计算评估指标
            metrics = calculate_ranking_metrics(masked_outputs, masked_targets, masks, k=5)
            for k, v in metrics.items():
                if k not in total_metrics:
                    total_metrics[k] = 0
                total_metrics[k] += v
            
            num_batches += 1
    
    # 计算平均指标
    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    for k in total_metrics:
        total_metrics[k] /= num_batches
    
    if writer:
        writer.add_scalar('eval/loss', avg_loss, global_step=epoch)
        for k, v in total_metrics.items():
            writer.add_scalar(f'eval/{k}', v, global_step=epoch)
    
    return avg_loss, total_metrics


def predict_top_stocks(model, data, features, sequence_length, scaler, stockid2idx, device, top_k=5):
    """
    预测某一天涨幅前top_k的股票
    """
    model.eval()
    
    # 获取最后一天的数据作为预测基础
    latest_date = data['日期'].max()
    
    # 准备预测数据
    day_sequences = []
    day_stock_codes = []
    day_stock_indices = []
    
    for stock_code in data['股票代码'].unique():
        # 获取该股票历史sequence_length天的数据
        stock_history = data[
            (data['股票代码'] == stock_code) & 
            (data['日期'] <= latest_date)
        ].sort_values('日期').tail(sequence_length)
        
        if len(stock_history) == sequence_length:
            seq = stock_history[features].values
            day_sequences.append(seq)
            day_stock_codes.append(stock_code)
            day_stock_indices.append(stockid2idx[stock_code])
    
    if len(day_sequences) == 0:
        return []
    
    # 转换为tensor
    sequences = torch.FloatTensor(np.array(day_sequences)).unsqueeze(0).to(device)  # [1, num_stocks, seq_len, features]
    
    with torch.no_grad():
        # 模型预测
        outputs = model(sequences)  # [1, num_stocks]
        scores = outputs.squeeze().cpu().numpy()  # [num_stocks]
        
        # 获取排名前top_k的股票
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        top_stocks = []
        for idx in top_indices:
            top_stocks.append({
                'stock_code': day_stock_codes[idx],
                'predicted_score': scores[idx],
                'rank': len(top_stocks) + 1
            })
    
    return top_stocks

def save_predictions(top_stocks, output_path):
    """保存预测结果"""
    results = []
    for stock in top_stocks:
        results.append({
            '排名': stock['rank'],
            '股票代码': stock['stock_code'],
            '预测分数': stock['predicted_score']
        })
    
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"预测结果已保存到: {output_path}")



#### 数据划分
- 用最后日期往前推 4 个月作为验证集起点 val_start
- 验证集额外往前取 sequence_length + max_feature_lookback - 1 个工作日做上下文，保证验证窗口可构造

In [11]:

def split_train_val_by_last_month(df, sequence_length):
    """按最后一个月做验证集划分，并为验证集补充序列上下文。"""
    df = df.copy()
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.sort_values(['日期', '股票代码']).reset_index(drop=True)

    last_date = df['日期'].max()
    val_start = (last_date - pd.DateOffset(months=2)).normalize()

    # 验证集需要保留前 sequence_length-1 个交易日作为序列上下文，
    # 这样第一个验证样本的窗口结束日就可以落在 val_start。
    max_feature_lookback = 80
    val_context_start = val_start - pd.tseries.offsets.BDay(sequence_length + max_feature_lookback - 1)

    train_df = df[df['日期'] < val_start].copy()
    val_df = df[df['日期'] >= val_context_start].copy()

    print(f"全量数据范围: {df['日期'].min().date()} 到 {last_date.date()}")
    print(f"训练集范围: {train_df['日期'].min().date()} 到 {train_df['日期'].max().date()}")
    print(f"验证集目标范围(最后一个月): {val_start.date()} 到 {last_date.date()}")
    print(f"验证集实际取数范围(含序列上下文): {val_df['日期'].min().date()} 到 {val_df['日期'].max().date()}")

    # 恢复为字符串，保持与原流程一致
    train_df['日期'] = train_df['日期'].dt.strftime('%Y-%m-%d')
    val_df['日期'] = val_df['日期'].dt.strftime('%Y-%m-%d')

    return train_df, val_df, val_start


#### 截面特征
- 对每天做 rank(pct) 和 zscore ，作为额外特征拼到 features。

In [12]:

def add_cross_sectional_features(df, base_cols):
    df = df.copy()
    df['日期'] = pd.to_datetime(df['日期'])

    new_cols = []

    for col in base_cols:
        if col not in df.columns:
            continue

        rank_col = f'{col}_cs_rank'
        z_col = f'{col}_cs_z'

        df[rank_col] = df.groupby('日期')[col].rank(pct=True)

        mean = df.groupby('日期')[col].transform('mean')
        std = df.groupby('日期')[col].transform('std')
        df[z_col] = (df[col] - mean) / (std + 1e-6)

        new_cols.extend([rank_col, z_col])

    df[new_cols] = df[new_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df, new_cols

def get_dataloader_predictions(model, dataloader, device):
    """辅助函数：获取 Transformer 对数据集的预测分数，用于 Stacking"""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            sequences = batch['sequences'].to(device)
            targets = batch['targets'].to(device)
            masks = batch['masks'].to(device)
            
            outputs = model(sequences, masks)
            
            # 过滤掉 padding 的无效股票
            for i in range(sequences.size(0)):
                mask = masks[i]
                valid_indices = mask.nonzero().squeeze()
                if valid_indices.dim() == 0:
                    valid_indices = valid_indices.unsqueeze(0)
                if valid_indices.numel() == 0:
                    continue
                
                all_preds.extend(outputs[i][valid_indices].cpu().numpy())
                all_targets.extend(targets[i][valid_indices].cpu().numpy())
                
    return np.array(all_preds), np.array(all_targets)

In [13]:
###-------------------------------添加包含提取最新截面特征功能的辅助函数-------------------------------
def get_dataloader_predictions_and_features(model, dataloader, device):
    """辅助函数：获取 Transformer 的预测分数，以及序列最后一天的截面特征，用于 Stacking"""
    model.eval()
    all_preds = []
    all_targets = []
    all_features = []
    
    with torch.no_grad():
        for batch in dataloader:
            sequences = batch['sequences'].to(device) # [batch, max_stocks, seq_len, features]
            targets = batch['targets'].to(device)
            masks = batch['masks'].to(device)
            
            outputs = model(sequences, masks)
            
            # 我们只需要序列的最后一天作为横截面特征给树模型
            # 形状变成: [batch, max_stocks, features]
            latest_features = sequences[:, :, -1, :] 
            
            # 过滤掉 padding 的无效股票
            for i in range(sequences.size(0)):
                mask = masks[i]
                valid_indices = mask.nonzero().squeeze()
                if valid_indices.dim() == 0:
                    valid_indices = valid_indices.unsqueeze(0)
                if valid_indices.numel() == 0:
                    continue
                
                all_preds.extend(outputs[i][valid_indices].cpu().numpy())
                all_targets.extend(targets[i][valid_indices].cpu().numpy())
                all_features.extend(latest_features[i][valid_indices].cpu().numpy())
                
    return np.array(all_preds), np.array(all_targets), np.array(all_features)

In [14]:
def get_dataloader_features_for_ranker(model, dataloader, device, feature_names, whitelist):
    """
    提取 Transformer 预测分数，并只提取 whitelist 中的特征。
    """
    model.eval()
    all_preds = []
    all_targets = []
    all_features = []
    all_groups = []
    all_rank_labels = []
    
    # 找出 whitelist 在 feature_names 中的索引
    # 同时必须把所有的 _cs_rank 和 _cs_z 特征也选进来
    selected_indices = []
    selected_feature_names = []
    for i, name in enumerate(feature_names):
        if name in whitelist or name.endswith('_cs_rank') or name.endswith('_cs_z'):
            selected_indices.append(i)
            selected_feature_names.append(name)
            
    print(f"LGBM 将使用 {len(selected_indices)} 个精简特征 (原特征数: {len(feature_names)})")
    
    with torch.no_grad():
        for batch in dataloader:
            sequences = batch['sequences'].to(device)
            targets = batch['targets'].to(device)
            masks = batch['masks'].to(device)
            
            outputs = model(sequences, masks)
            # 取最后一天，并只保留选中的特征维度
            latest_features = sequences[:, :, -1, selected_indices] 
            
            batch_size = sequences.size(0)
            for i in range(batch_size):
                mask = masks[i]
                valid_indices = mask.nonzero().squeeze()
                if valid_indices.dim() == 0:
                    valid_indices = valid_indices.unsqueeze(0)
                
                num_valid_stocks = valid_indices.numel()
                if num_valid_stocks == 0:
                    continue
                
                day_preds = outputs[i][valid_indices].cpu().numpy()
                day_targets = targets[i][valid_indices].cpu().numpy()
                day_features = latest_features[i][valid_indices].cpu().numpy()
                
                sorted_idx = np.argsort(day_targets)[::-1]
                day_labels = np.zeros_like(day_targets, dtype=int)
                for rank, idx in enumerate(sorted_idx):
                    if rank < 5:
                        day_labels[idx] = 4
                    elif rank < 20:
                        day_labels[idx] = 3
                    elif rank < 50:
                        day_labels[idx] = 2
                    elif rank < int(num_valid_stocks * 0.3): 
                        day_labels[idx] = 1
                    else:
                        day_labels[idx] = 0
                
                all_groups.append(num_valid_stocks)
                all_preds.extend(day_preds)
                all_targets.extend(day_targets)
                all_features.extend(day_features)
                all_rank_labels.extend(day_labels)
                
    return (np.array(all_preds), np.array(all_targets), 
            np.array(all_features), np.array(all_groups), np.array(all_rank_labels), selected_feature_names)

#### 主程序



In [15]:

# 主程序
def main():
    set_seed(config.get('seed', 42))
    output_dir = config['output_dir']
    os.makedirs(output_dir,exist_ok=True)
    # 保存在output_dir中保存当前的配置文件，以便复现
    data_path = config['data_path']
    with open(os.path.join(output_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=4, ensure_ascii=False)
    is_train = True
    writer = SummaryWriter(log_dir=os.path.join(output_dir, 'log')) if is_train else None
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    
    # === 新增: LGBM Stacking 的专属特征白名单 ===
    # 基于特征重要性挑选：波动率、换手率、长周期(30/60)、成交量特征，以及所有的横截面特征(cs_z, cs_rank)
    lgbm_feature_whitelist = [
        'volume_ma_20', 'volume_ma_5', 'obv', 'volatility_20', 'volatility_10',
        'WVMA60', 'MIN60', 'BETA60', 'STD60', 'WVMA10', 'VSUMP60', 'STD30', 'MIN30', 'WVMA20',
        'MA60', 'MAX60', 'RSV60', 'VMA60', 'VSTD60', 'WVMA30', 'VSUMN60', 'VSUMD60',
        'return_1', 'return_5', 'return_10', 'rsi', 'macd', 'atr_14', '换手率', '涨跌幅', '成交量', '成交额'
    ]
    
    # 1. 数据加载
    data_file = os.path.join(config['data_path'], 'stock_data.csv')
    # full_df = pd.read_csv(data_file)
    full_df = pd.read_csv(data_file, dtype={'股票代码': str})
    full_df['股票代码'] = full_df['股票代码'].astype(str).str.zfill(6)
    train_df, val_df, val_start = split_train_val_by_last_month(full_df, config['sequence_length'])
    
    # 获取所有股票ID，建立映射
    all_stock_ids = full_df['股票代码'].unique()
    stockid2idx = {sid: idx for idx, sid in enumerate(sorted(all_stock_ids))}
    num_stocks = len(stockid2idx)
    
    # 2. 特征工程与预处理
    train_data, features = preprocess_data(train_df, is_train=True, stockid2idx=stockid2idx)
    val_data, _ = preprocess_val_data(val_df, stockid2idx=stockid2idx)
    
    cs_base_cols = [
    '涨跌幅',
    '换手率',
    '成交额',
    '成交量',
    'return_1',
    'return_5',
    'return_10',
    'volume_ratio',
    'volatility_10',
    'volatility_20',
    'rsi',
    'macd',
    'atr_14'
    ]

    train_data, cs_features = add_cross_sectional_features(train_data, cs_base_cols)
    val_data, _ = add_cross_sectional_features(val_data, cs_base_cols)

    features = features + cs_features
    features = list(dict.fromkeys(features))  # 去重

    # 3. 标准化
    scaler = StandardScaler()

    train_data[features] = train_data[features].replace([np.inf, -np.inf], np.nan)
    val_data[features] = val_data[features].replace([np.inf, -np.inf], np.nan)
    # 丢弃nan数据
    train_data = train_data.dropna(subset=features)
    val_data = val_data.dropna(subset=features)
    # 然后再缩放
    train_data[features] = scaler.fit_transform(train_data[features])
    val_data[features] = scaler.transform(val_data[features])
    joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))

    
    # 4. 创建排序数据集
    train_sequences, train_targets, train_relevance, train_stock_indices = create_ranking_dataset_vectorized(
        train_data,
        features,
        config['sequence_length'],
        ranking_data_path=config.get('train_ranking_data_path')
    )
    val_sequences, val_targets, val_relevance, val_stock_indices = create_ranking_dataset_vectorized(
        val_data,
        features,
        config['sequence_length'],
        ranking_data_path=config.get('val_ranking_data_path'),
        min_window_end_date=val_start.strftime('%Y-%m-%d')
    )

    print(f"训练集样本数: {len(train_sequences)}")
    print(f"验证集样本数: {len(val_sequences)}")
    
    # 5. 创建排序数据集和数据加载器
    train_dataset = RankingDataset(train_sequences, train_targets, train_relevance, train_stock_indices)
    val_dataset = RankingDataset(val_sequences, val_targets, val_relevance, val_stock_indices)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True, 
        collate_fn=collate_fn,
        num_workers=0,  # 减少worker数量避免内存问题
        pin_memory=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config['batch_size'], 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=False
    )
    
    # 6. 模型初始化
    model = StockTransformer(input_dim=len(features), config=config, num_stocks=num_stocks)
    model.to(device)
    print(f"模型参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
    
    # 7. 损失函数和优化器
    criterion = WeightedRankingLoss(
        k=5,
        temperature=config.get('temperature', 1.0),
        weight_factor=config['top5_weight'],
        pairwise_weight=config['pairwise_weight'],
        base_weight=config.get('base_weight', 1.0)
    )  # 使用加权排序损失
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.2, total_iters=config['num_epochs'])
    
    patience = config.get('patience', 8)
    bad_epochs = 0
    best_score = -float('inf')
    best_epoch = -1


    if is_train:
        from sklearn.model_selection import KFold
        import lightgbm as lgb
        
        # ========================================================
        # 阶段 1: Transformer K-Fold OOF (提取无偏的时序预测分数)
        # ========================================================
        print("\n========== 阶段 1: Transformer K-Fold OOF ==========")
        
        # 【关键修改 1】绝对不能在时间序列中打乱数据！
        # 因为 train_sequences 已经是按时间顺序排列的，我们按时间块划分 OOF
        kf = KFold(n_splits=3, shuffle=False)
        
        train_seq_np = np.array(train_sequences, dtype=object)
        train_tgt_np = np.array(train_targets, dtype=object)
        train_rel_np = np.array(train_relevance, dtype=object)
        train_idx_np = np.array(train_stock_indices, dtype=object)
        
        oof_train_preds = []
        oof_train_features = []
        oof_train_groups = []
        oof_train_rank_labels = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_seq_np)):
            print(f"\n--- Fold {fold+1}/5 ---")
            fold_train_dataset = RankingDataset(train_seq_np[train_idx], train_tgt_np[train_idx], train_rel_np[train_idx], train_idx_np[train_idx])
            fold_val_dataset = RankingDataset(train_seq_np[val_idx], train_tgt_np[val_idx], train_rel_np[val_idx], train_idx_np[val_idx])
            
            fold_train_loader = DataLoader(fold_train_dataset, batch_size=config['batch_size'], shuffle=True, collate_fn=collate_fn)
            fold_val_loader = DataLoader(fold_val_dataset, batch_size=config['batch_size'], shuffle=False, collate_fn=collate_fn)
            
            fold_model = StockTransformer(input_dim=len(features), config=config, num_stocks=num_stocks).to(device)
            fold_optimizer = torch.optim.AdamW(fold_model.parameters(), lr=config['learning_rate'], weight_decay=1e-5)
            
            # 为节省时间，Fold 内只训练 3 个 Epoch 提取特征
            fold_epochs = 1
            for epoch in range(fold_epochs):
                train_ranking_model(fold_model, fold_train_loader, criterion, fold_optimizer, device, epoch, None)
            
            #  获取 OOF 预测分数和对应的 Group、Label
            # fold_preds, _, fold_features, fold_groups, fold_rank_labels = \
            #     get_dataloader_features_for_ranker(fold_model, fold_val_loader, device)
                        # 获取 OOF 预测分数和对应的 Group、Label (注意这里加了参数)
            fold_preds, _, fold_features, fold_groups, fold_rank_labels, selected_feature_names = \
                get_dataloader_features_for_ranker(fold_model, fold_val_loader, device, features, lgbm_feature_whitelist)

            oof_train_preds.extend(fold_preds)
            oof_train_features.extend(fold_features)
            oof_train_groups.extend(fold_groups)
            oof_train_rank_labels.extend(fold_rank_labels)
            
        oof_train_preds = np.array(oof_train_preds)
        oof_train_features = np.array(oof_train_features)
        oof_train_groups = np.array(oof_train_groups)
        oof_train_rank_labels = np.array(oof_train_rank_labels)
        
        # ========================================================
        # 阶段 2: 训练全量 Transformer (作为主模型)
        # ========================================================
        print("\n========== 阶段 2: 训练全量 Transformer ==========")
        best_score = -float('inf')
        best_epoch = -1
        bad_epochs = 0
        
        for epoch in range(config['num_epochs']):
            print(f"\n=== Epoch {epoch+1}/{config['num_epochs']} ===")
            train_loss, train_metrics = train_ranking_model(model, train_loader, criterion, optimizer, device, epoch, writer)
            eval_loss, eval_metrics = evaluate_ranking_model(model, val_loader, criterion, device, writer, epoch)
            scheduler.step()
            
            current_final_score = eval_metrics.get('final_score', 0.0)
            if current_final_score > best_score:
                best_score = current_final_score
                best_epoch = epoch + 1
                bad_epochs = 0
                torch.save(model.state_dict(), os.path.join(output_dir, 'best_model.pth'))
                print(f"保存最佳模型 - final score: {best_score:.4f}")
            else:
                bad_epochs += 1
                print(f"验证集未提升: {bad_epochs}/{patience}")

            if bad_epochs >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
                
        # ========================================================
        # 阶段 3: LGBMRanker Stacking (目标一致：均为 Ranking)
        # ========================================================
        print("\n========== 阶段 3: LGBMRanker Stacking ==========")
        
        # 1. 构建 Stacking 训练集：[全部截面特征, Transformer 分数]
        stacking_X_train = np.hstack([oof_train_features, oof_train_preds.reshape(-1, 1)])
        stacking_y_train = oof_train_rank_labels  
        stacking_group_train = oof_train_groups   
        

        model.load_state_dict(torch.load(os.path.join(output_dir, 'best_model.pth')))
        val_preds, _, val_features, val_groups, val_rank_labels, _ = \
            get_dataloader_features_for_ranker(model, val_loader, device, features, lgbm_feature_whitelist)

        stacking_X_val = np.hstack([val_features, val_preds.reshape(-1, 1)])
        stacking_y_val = val_rank_labels
        stacking_group_val = val_groups
        
        # 【关键修改 2】增强 LGBM 的抗过拟合能力
        ranker_params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'ndcg_eval_at': [5, 20],
            'n_estimators': 500,         # 增加树的数量
            'learning_rate': 0.01,       # 降低学习率，防止过早收敛 (原为 0.05)
            'max_depth': 5,              # 降低树深，防止死记硬背 (原为 6)
            'num_leaves': 15,            # 降低叶子节点数 (原为 31)
            'colsample_bytree': 0.6,     # 每次分裂只用 60% 的特征 (原为 0.8)
            'subsample': 0.6,            # 行采样 (原为 0.8)
            'min_child_samples': 50,     # 叶子节点最少样本数，强效防过拟合
            'random_state': 42,
            'n_jobs': -1,
            'label_gain': [0, 1, 3, 7, 15] 
        }
        
        lgb_ranker = lgb.LGBMRanker(**ranker_params)
        
        lgb_ranker.fit(
            stacking_X_train, stacking_y_train,
            group=stacking_group_train,
            eval_set=[(stacking_X_val, stacking_y_val)],
            eval_group=[stacking_group_val],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)] # 增加早停容忍度
        )
        
        joblib.dump(lgb_ranker, os.path.join(output_dir, 'lgb_stacking_ranker.pkl'))
        print("\nLGBMRanker 训练完成，模型已保存！")
        
        # 【关键修改 3】打印出真实的特征名称，帮助你做特征筛选
        # 获取所有原始特征的名称列表
        all_feature_names = selected_feature_names.copy() 
        all_feature_names.append("Transformer_Score") # 加上最后一列
        
        importances = lgb_ranker.feature_importances_
        indices = np.argsort(importances)[::-1]
        print("\nTop 20 重要特征 (真实名称):")
        for f in range(min(20, len(all_feature_names))): # 防止越界
            idx = indices[f]
            feature_name = all_feature_names[idx] if idx < len(all_feature_names) else f"Unknown_{idx}"
            print(f"{f+1}. {feature_name} (重要性: {importances[idx]})")
              
        # 善后工作
        print(f"\n全流程完成！Transformer最佳 epoch: {best_epoch}, 最佳 final score: {best_score:.4f}")
        with open(os.path.join(output_dir, 'final_score.txt'), 'w') as f:
            f.write(f"Best epoch: {best_epoch}\nBest final_score: {best_score:.6f}\n")

        if writer:
            writer.close()

        return best_score
best_score = main()
print(f"\n########## 训练完成！最佳 final score: {best_score:.4f} ##########")

全量数据范围: 2024-10-08 到 2026-04-15
训练集范围: 2024-10-08 到 2026-02-13
验证集目标范围(最后一个月): 2026-02-15 到 2026-04-15
验证集实际取数范围(含序列上下文): 2025-08-05 到 2026-04-15
正在进行特征工程，当前使用串行模式，避免 Notebook 多进程卡死...


特征工程:   3%|▎         | 8/300 [00:00<00:17, 16.46it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['volume_change'] = volume.pct_change()
特征工程:  13%|█▎        | 40/300 [00:02<00:14, 18.38it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['volume_change'] = volume.pct_change()
特征工程:  16%|█▌        | 48/300 [00:02<00:13, 18.20it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fi

正在进行验证集特征工程，当前使用串行模式，避免 Notebook 多进程卡死...


验证集特征工程:  13%|█▎        | 40/300 [00:01<00:12, 21.47it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['volume_change'] = volume.pct_change()
验证集特征工程:  30%|███       | 90/300 [00:04<00:10, 19.73it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['volume_change'] = volume.pct_change()
验证集特征工程:  45%|████▌     | 135/300 [00:06<00:07, 21.57it/s]<user>:109: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None

正在创建排序数据集（向量化加速版本）...
Step 1: 为每只股票生成滑动窗口...


...

验证集未提升: 9/10

=== Epoch 13/40 ===


Evaluating Epoch 13: 100%|██████████| 7/7 [00:00<00:00, 20.19it/s]


验证集未提升: 10/10
Early stopping at epoch 13

========== 阶段 3: LGBMRanker Stacking ==========
LGBM 将使用 58 个精简特征 (原特征数: 222)


site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007428 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15033
[LightGBM] [Info] Number of data points in the train set: 79252, number of used features: 59
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's ndcg@5: 0.133258	valid_0's ndcg@20: 0.226826

LGBMRanker 训练完成，模型已保存！

Top 20 重要特征 (真实名称):
1. Transformer_Score (重要性: 9)
2. volatility_20_cs_z (重要性: 8)
3. STD60 (重要性: 5)
4. WVMA60 (重要性: 4)
5. MAX60 (重要性: 4)
6. atr_14 (重要性: 4)
7. STD30 (重要性: 3)
8. rsi_cs_rank (重要性: 3)
9. MIN30 (重要性: 3)
10. obv (重要性: 3)
11. MA60 (重要性: 2)
12. 成交量_cs_z (重要性: 2)
13. MIN60 (重要性: 2)
14. RSV60 (重要性: 2)
15. WVMA20 (重要性: 2)
16. 成交额 (重要性: 2)
17. VSUMN60 (重要性: 2)
18. 成交量 (重要性: 2)
19. volatility_20_cs_rank (重要性: 2)
20. volatility_10 (重要性: 1)

全流程完成！Transformer最佳 epoch: 3, 最佳 final score: 0.0899

########## 训练完成！最佳 final sco

## 06模型预测
- 这一段是从训练流程复用的预测步骤：特征工程 → scaler 变换 → 构造最后一天所有股票序列 → 加载模型 → 输出 Top-5 并写 CSV。
- 当前已注释掉，notebook 运行只做训练与验证；提交时取消注释并按 `predict_top_stocks` / `save_predictions` 的接口生成结果文件。